# Item-based CF - похожие товары по совместным покупкам, KNN

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
import hdbscan
from sklearn.manifold import TSNE
import umap
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
import openpyxl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.compose import ColumnTransformer
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

In [2]:
#df - транзакционный очищенный датасет + кластеры каждого пользователя
#client_df - агрегированный датасет по пользователям
df = pd.read_parquet('df.parquet', engine='fastparquet')
client_df = pd.read_parquet('client_data.parquet', engine='fastparquet')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1393343 entries, 0 to 1393342
Data columns (total 40 columns):
 #   Column                    Non-Null Count    Dtype         
---  ------                    --------------    -----         
 0   Дата                      1393343 non-null  datetime64[ns]
 1   ДатаДоставки              1393343 non-null  datetime64[ns]
 2   НомерЗаказаНаСайте        1393343 non-null  object        
 3   НовыйСтатус               1393343 non-null  category      
 4   СуммаЗаказаНаСайте        1393343 non-null  float64       
 5   СуммаДокумента            1393343 non-null  float64       
 6   МетодДоставки             1393343 non-null  category      
 7   ФормаОплаты               1393343 non-null  category      
 8   Регион                    1379232 non-null  category      
 9   Группа2                   1393343 non-null  category      
 10  Группа3                   1393343 non-null  category      
 11  Группа4                   1335893 non-null  catego

In [7]:
#обрезаем для проверки кода
df=df.head(10000)
valid_phones = df['Телефон_new'].unique()
client_df = client_df[client_df['Телефон_new'].isin(valid_phones)].copy()

In [11]:
# Создаем train и test (последняя покупка каждого пользователя - в test)
df_sorted = df.sort_values(['Телефон_new', 'ДатаЗаказаНаСайте'])
train_df = df_sorted.groupby('Телефон_new').apply(lambda x: x.iloc[:-1]).reset_index(drop=True)
test_df = df_sorted.groupby('Телефон_new').apply(lambda x: x.iloc[-1:]).reset_index(drop=True)

print(f"Train размер: {len(train_df)}")
print(f"Test размер: {len(test_df)}")
print(f"Уникальных пользователей в train: {train_df['Телефон_new'].nunique()}")
print(f"Уникальных товаров в train: {train_df['Номенклатура'].nunique()}")

/var/folders/21/d9w0s1wn3y772gh03j1bs5wm0000gn/T/ipykernel_46067/2179642130.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df = df_sorted.groupby('Телефон_new').apply(lambda x: x.iloc[:-1]).reset_index(drop=True)


Train размер: 6220
Test размер: 3780
Уникальных пользователей в train: 1780
Уникальных товаров в train: 4227


/var/folders/21/d9w0s1wn3y772gh03j1bs5wm0000gn/T/ipykernel_46067/2179642130.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_df = df_sorted.groupby('Телефон_new').apply(lambda x: x.iloc[-1:]).reset_index(drop=True)


In [12]:
# Выбираем топ популярных товаров для уменьшения размера матрицы
TOP_ITEMS = 3000 
print(f"Выбираем топ-{TOP_ITEMS} популярных товаров")

popular_items = train_df['Номенклатура'].value_counts().head(TOP_ITEMS).index.tolist()
print(f"Уникальных товаров после фильтрации: {len(popular_items)}")

# Фильтруем train только популярные товары
train_filtered = train_df[train_df['Номенклатура'].isin(popular_items)]
print(f"Train после фильтрации: {len(train_filtered)} записей")

# Создаем маппинги для пользователей и товаров
users = train_filtered['Телефон_new'].unique()
items = popular_items

user_to_idx = {user: idx for idx, user in enumerate(users)}
item_to_idx = {item: idx for idx, item in enumerate(items)}
idx_to_item = {idx: item for item, idx in item_to_idx.items()}

print(f"Уникальных пользователей: {len(users):,}")
print(f"Уникальных товаров: {len(items):,}")

Выбираем топ-3000 популярных товаров
Уникальных товаров после фильтрации: 3000
Train после фильтрации: 4993 записей
Уникальных пользователей: 1,541
Уникальных товаров: 3,000


In [14]:
# Создаем списки для sparse матрицы
rows = [user_to_idx[user] for user in train_filtered['Телефон_new']]
cols = [item_to_idx[item] for item in train_filtered['Номенклатура']]
data = [1] * len(rows)  # бинарные взаимодействия (1 - купил)

# Создаем csr матрицу
user_item_matrix = csr_matrix((data, (rows, cols)), 
                               shape=(len(users), len(items)))

In [15]:
# Транспонируем для получения item-user матрицы (товары x пользователи)
item_user_matrix = user_item_matrix.T
print(f"Размер item-user матрицы: {item_user_matrix.shape}")

Размер item-user матрицы: (3000, 1541)


In [17]:
# Настраиваем KNN
N_NEIGHBORS = 100  # Ищем 100 ближайших соседей
knn = NearestNeighbors(n_neighbors=N_NEIGHBORS, metric='cosine', algorithm='brute')

# Обучаем на item-user матрице (товары x пользователи)
knn.fit(item_user_matrix)

,n_neighbors,100
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [18]:
# Словарь для хранения похожих товаров
similar_items_dict = {}

for item_idx in tqdm(range(len(items)), desc="Поиск похожих товаров"):
    # Получаем вектор товара
    item_vector = item_user_matrix[item_idx].reshape(1, -1)
    
    # Находим ближайших соседей
    distances, indices = knn.kneighbors(item_vector, n_neighbors=N_NEIGHBORS)
    
    # Сохраняем похожие товары (исключая сам товар)
    similar_items_dict[item_idx] = indices[0][1:].tolist()
print(f"Каждый товар имеет {len(similar_items_dict[0])} похожих товаров")

Поиск похожих товаров: 100%|██████████████| 3000/3000 [00:01<00:00, 2769.18it/s]

Каждый товар имеет 99 похожих товаров


In [19]:
def recommend_item_based(user_id, k=10):
    """
    Рекомендации на основе Item-based Collaborative Filtering с KNN
    """
    # Проверяем, есть ли пользователь
    if user_id not in user_to_idx:
        # Для новых пользователей - популярные товары
        return popular_items[:k]
    
    user_idx = user_to_idx[user_id]
    
    # Получаем товары, которые пользователь уже купил
    user_items = user_item_matrix[user_idx].indices
    user_items_set = set(user_items)
    
    if len(user_items) == 0:
        return popular_items[:k]
    
    # Собираем scores для всех товаров
    scores = {}
    
    # Для каждого купленного товара
    for item_idx in user_items:
        # Получаем похожие товары из словаря
        similar_items = similar_items_dict.get(item_idx, [])
        
        # Добавляем scores (каждый похожий товар получает +1)
        for sim_idx in similar_items:
            if sim_idx not in user_items_set:
                scores[sim_idx] = scores.get(sim_idx, 0) + 1
    
    # Если нет подходящих товаров, возвращаем популярные
    if not scores:
        return popular_items[:k]
    
    # Сортируем по score и берем топ-k
    recommended_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    
    # Преобразуем индексы в названия товаров
    recommendations = [idx_to_item[item_idx] for item_idx, _ in recommended_items]
    
    return recommendations

# Тестируем
test_user = users[0]
print(f"Рекомендации для пользователя {test_user}:")
recs = recommend_item_based(test_user, k=5)
print(f"Рекомендации: {recs}")

Рекомендации для пользователя 55574848-51494850484970:
Рекомендации: ['ФАБРИКА ФАНТАЗИЙ, РАМКА из дерева Сова', 'PELICAN, КОМПЛЕКТ (футболка+шорты), (зелен), р.128', 'MAMATTI, ШТАНИШКИ Bird, р. 68, Польша', 'LEADER KIDS, ЧЕПЧИК Первая зима, (роз), р. 56', 'LEADER KIDS, САРАФАН (сер), р. 74']


In [20]:
def evaluate_item_based(test_df, recommender_func, k=10):
    """
    Оценка Item-based CF модели
    """
    hits = []
    precisions = []
    recalls = []
    average_precisions = []
    mrr = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Оценка (k={k})"):
        user = row['Телефон_new']
        true_item = row['Номенклатура']
        
        try:
            recs = recommender_func(user, k)
        except Exception as e:
            continue
        
        hit = int(true_item in recs)
        hits.append(hit)
        precisions.append(hit / k)
        recalls.append(hit)
        
        if hit:
            rank = recs.index(true_item) + 1
            average_precisions.append(1 / rank)
            mrr.append(1 / rank)
        else:
            average_precisions.append(0)
            mrr.append(0)
    
    return {
        "HitRate@K": np.mean(hits),
        "Precision@K": np.mean(precisions),
        "Recall@K": np.mean(recalls),
        "MAP@K": np.mean(average_precisions),
        "MRR@K": np.mean(mrr),
        "Test_size": len(test_df)
    }

In [21]:
metrics_item_5 = evaluate_item_based(test_df, recommend_item_based, k=5)
metrics_item_10 = evaluate_item_based(test_df, recommend_item_based, k=10)
metrics_item_20 = evaluate_item_based(test_df, recommend_item_based, k=20)

print("\n" + "="*50)
print("Item-based CF модель (с KNN):")
print("="*50)
print(f"K=5: {metrics_item_5}")
print(f"K=10: {metrics_item_10}")
print(f"K=20: {metrics_item_20}")

Оценка (k=20): 100%|█████████████████████| 3780/3780 [00:00<00:00, 35267.73it/s]


Item-based CF модель (с KNN):
K=5: {'HitRate@K': np.float64(0.019312169312169312), 'Precision@K': np.float64(0.0038624338624338624), 'Recall@K': np.float64(0.019312169312169312), 'MAP@K': np.float64(0.010745149911816578), 'MRR@K': np.float64(0.010745149911816578), 'Test_size': 3780}
K=10: {'HitRate@K': np.float64(0.025396825396825397), 'Precision@K': np.float64(0.00253968253968254), 'Recall@K': np.float64(0.025396825396825397), 'MAP@K': np.float64(0.0115360712186109), 'MRR@K': np.float64(0.0115360712186109), 'Test_size': 3780}
K=20: {'HitRate@K': np.float64(0.0335978835978836), 'Precision@K': np.float64(0.00167989417989418), 'Recall@K': np.float64(0.0335978835978836), 'MAP@K': np.float64(0.012083101046686482), 'MRR@K': np.float64(0.012083101046686482), 'Test_size': 3780}


In [22]:
print("\n" + "="*60)
print("АНАЛИЗ ПОКРЫТИЯ ДЛЯ ITEM-BASED CF")
print("="*60)

test_items = test_df['Номенклатура'].unique()
train_items = train_filtered['Номенклатура'].unique()
new_items = set(test_items) - set(train_items)
common_items = set(test_items) & set(train_items)

print(f"Всего товаров в тесте: {len(test_items)}")
print(f"Новых товаров (нет в train): {len(new_items)} ({len(new_items)/len(test_items)*100:.1f}%)")
print(f"Общих товаров (может предсказать): {len(common_items)} ({len(common_items)/len(test_items)*100:.1f}%)")
print(f"Товаров в топ-{TOP_ITEMS}: {len([x for x in test_items if x in popular_items])} из {len(test_items)}")
print("="*60)


АНАЛИЗ ПОКРЫТИЯ ДЛЯ ITEM-BASED CF
Всего товаров в тесте: 2703
Новых товаров (нет в train): 2172 (80.4%)
Общих товаров (может предсказать): 531 (19.6%)
Товаров в топ-3000: 531 из 2703


In [23]:
!jupyter nbconvert --to script ВКР_Zinoveva_3_item_based.ipynb

[NbConvertApp] Converting notebook ВКР_Zinoveva_3_item_based.ipynb to script
[NbConvertApp] Writing 8292 bytes to ВКР_Zinoveva_3_item_based.py
